In [ ]:
import sqlite3
import pandas as pd
import plotly.express as px
from PIL import Image
import matplotlib.pyplot as plt


In [ ]:
def plotids(selected_ids):
    import numpy as np
    import matplotlib.pyplot as plt
    import tables
    print(f"Selected {len(selected_ids)} points")
    n = min(len(selected_ids), 100)
    cols = 10
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(15, 1.5 * rows))
    axes = np.array(axes).flat  # handles both 1-row and multi-row cases

    with tables.open_file("mitosis_train_patches.pytable", "r") as f:
        patches = f.root.patch[selected_ids[:n],::]
        labels = f.root.label_jc[selected_ids[:n]]
        patch_ids = f.root.patch_id_unique[selected_ids[:n]]
        

    for i, ax in enumerate(axes):
        if i < n:
            ax.imshow(patches[i])
            ax.set_title(f"id:{patch_ids[i]}\nlbl:{labels[i]}", fontsize=6)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# --- Load coordinates from SpatiaLite ---
conn = sqlite3.connect("coords_embeddings.db")
df  = pd.read_sql("SELECT id, x, y,emb FROM patches", conn)


In [ ]:
df

In [ ]:
import tables
with tables.open_file("mitosis_train_patches.pytable", "r") as f:
    all_labels = f.root.label_jc[:]
    

In [ ]:
#----

In [ ]:
import os, sqlite_vector
conn.enable_load_extension(True)
so_path = os.path.join(os.path.dirname(sqlite_vector.__file__), "binaries", "vector")
conn.load_extension(so_path)


In [ ]:
# -- Initialize the vector. By default, the distance function is L2.
# -- To use a different metric, specify one of the following options:
# -- distance=L1, distance=COSINE, distance=DOT, distance=SQUARED_L2, or distance=HAMMING.
# SELECT vector_init('images', 'embedding', 'type=FLOAT32,dimension=384');
conn.execute("SELECT vector_init('patches', 'emb', 'type=FLOAT32,dimension=16')")

In [ ]:
# import numpy as np
# df["x"] = np.random.rand(len(df))
# df["y"] = np.random.rand(len(df))


In [ ]:
df["x"]

In [ ]:
import plotly.graph_objs as go

fw = go.FigureWidget(data=go.Scatter(
    x=df["x"], y=df["y"],
    mode="markers",
    customdata=df["id"],
))
fw.layout.dragmode = "lasso"

selected_ids = []

def on_select(trace, points, selector):
    selected_ids.clear()
    selected_ids.extend([trace.customdata[i] for i in points.point_inds])
    print(f"Selected {len(selected_ids)} points")

fw.data[0].on_selection(on_select)
fw

In [ ]:
len(selected_ids)

In [ ]:
selected_ids

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import tables
print(f"Selected {len(selected_ids)} points")
n = min(len(selected_ids), 100)
cols = 10
rows = (n + cols - 1) // cols


In [ ]:

fig, axes = plt.subplots(rows, cols, figsize=(15, 1.5 * rows))
axes = np.array(axes).flat  # handles both 1-row and multi-row cases

f=tables.open_file("mitosis_ps_labels.pytable", "r")


In [ ]:
f.root.patch.shape

In [ ]:
patches = f.root.patch[selected_ids[:n],::]


In [ ]:

labels = f.root.label_jc[selected_ids[:n]]
#    patch_ids = f.root.patch_id[selected_ids[:n]]



In [ ]:

for i, ax in enumerate(axes):
    if i < n:
        ax.imshow(patches[i])
        ax.set_title(f"id:{patch_ids[i]}\nlbl:{labels[i]}", fontsize=6)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plotids(selected_ids)


In [ ]:
#--- high dimensional search

In [ ]:
%%time
def query_vector(mean_emb):
    query_blob = sqlite3.Binary(mean_emb.tobytes())
    results = conn.execute("""
        SELECT p.id, v.distance
        FROM patches AS p
        JOIN vector_full_scan('patches', 'emb', ?) AS v ON p.id = v.rowid
        ORDER BY v.distance
        LIMIT 100
    """, (query_blob,)).fetchall()
    ids, distances = zip(*results)
    print(results[0:5])
    return list(ids), list(distances)
def query_id(id):
    query_vec = np.frombuffer(df.loc[id, "emb"], dtype=np.float32)
    return query_vector(query_vec)


In [ ]:
import numpy as np
embs = np.stack([np.frombuffer(e, dtype=np.float32) for e in df.loc[selected_ids, "emb"]])
mean_emb = embs.mean(axis=0)
results = query_vector(mean_emb)

In [ ]:
# --- quantization --- but would focus on full reprensetations first. be ware, need to check that the results are meaningful  with q=2 + 4 the results were very suboptimal
# #-- seutp quantization (for faster search, with some loss in accuracy)
# conn.execute("SELECT vector_quantize('patches', 'emb', 'qtype=TURBO,qbits=4');")
# ##-- Optional preload quantized version in memory (for a 4x/5x speedup) 
# conn.execute("SELECT vector_quantize_preload('patches', 'emb');")
# %%time
# query_blob = sqlite3.Binary(mean_emb.tobytes())
# results = conn.execute("""
#     SELECT p.id, v.distance
#     FROM patches AS p
#     JOIN vector_quantize_scan('patches', 'emb', ?) AS v ON p.id = v.rowid
#     ORDER BY v.distance
#     LIMIT 100
# """, (query_blob,)).fetchall()

# top_ids = [r[0] for r in results]
# #print(top_ids[:5])
# print(results[0:5])



In [ ]:
##sanity check
print(np.frombuffer(df.loc[results[0][0],"emb"],dtype=np.float32))
print(mean_emb)

In [ ]:
plotids(results[0])

In [ ]:
plotids((all_labels==1).nonzero()[0])

In [ ]:
posids=(all_labels==1).nonzero()[0]
pid=1
print(posids[pid])
plotids(query_id(posids[pid])[0])